# Можно ли предсказать стоимость жилья по характеристикам района?

**Небольшой учебный эксперимент по регрессии.** Я использую открытый California Housing (перепись 1990 года). Сначала посмотрю на данные и простую модель, затем попробую два более сложных алгоритма.

Важная деталь: наблюдения относятся к географическим точкам, поэтому для тестирования лучше отделять целые участки карты, а не случайные строки.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df.head()

## 1. Проверяю данные

Цель — `MedHouseVal`, стоимость в единицах по 100 000 долларов. В датасете 8 числовых признаков. Проверю пропуски и самые простые связи.

In [ ]:
print('Размер:', df.shape)
print()
print('Пропуски:')
print(df.isna().sum())
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df['MedHouseVal'].hist(bins=35, ax=axes[0])
axes[0].set_title('Распределение стоимости')
axes[0].set_xlabel('Стоимость, x 100 000 $')
axes[1].scatter(df['MedInc'], df['MedHouseVal'], s=5, alpha=0.2)
axes[1].set_title('Доход района и стоимость жилья')
axes[1].set_xlabel('MedInc')
axes[1].set_ylabel('MedHouseVal')
plt.tight_layout()

## 2. Деление на выборки

Вместо случайного разделения использую сетку по широте и долготе (шаг 0.5°). Ячейки теста не должны попадать в обучение. Соседние ячейки всё равно могут быть похожи — это ограничение метода.

In [ ]:
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']
groups = (np.floor(X['Latitude'] / 0.5).astype(int).astype(str) + '_' +
          np.floor(X['Longitude'] / 0.5).astype(int).astype(str))
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[test_idx])
print(f'train={len(train_idx)}, test={len(test_idx)}')

## 3. Сравнение моделей

Сначала предсказываю медиану — это baseline. Затем сравниваю Ridge и Random Forest по RMSE. Масштабирование и заполнение пропусков находятся внутри Pipeline, чтобы не использовать статистики из теста.

In [ ]:
models = {
    'baseline': make_pipeline(SimpleImputer(strategy='median'), DummyRegressor(strategy='median')),
    'ridge': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=2.0)),
    'random_forest': make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(
        n_estimators=80, min_samples_leaf=3, random_state=42, n_jobs=-1)),
}
cv = GroupKFold(n_splits=3)
results = {}
for name, model in models.items():
    scores = -cross_val_score(model, X_train, y_train, groups=groups.iloc[train_idx],
                              cv=cv, scoring='neg_root_mean_squared_error', n_jobs=1)
    results[name] = scores.mean()
    print(name, 'CV RMSE:', round(scores.mean(), 3))

## 4. Финальная проверка

Выбираю модель по CV, **не по тестовой выборке**. Test нужен для окончательной проверки, а не для многократной настройки.

In [ ]:
winner = min(results, key=results.get)
final_model = models[winner].fit(X_train, y_train)
pred = final_model.predict(X_test)
print('Выбрана модель:', winner)
print('Test MAE:', round(mean_absolute_error(y_test, pred), 3))
print('Test RMSE:', round(np.sqrt(mean_squared_error(y_test, pred)), 3))
print('Test R2:', round(r2_score(y_test, pred), 3))
plt.figure(figsize=(6, 4))
plt.scatter(y_test, pred, s=6, alpha=0.3)
plt.plot([0, 5], [0, 5], 'k--', linewidth=1)
plt.xlabel('Истинное значение')
plt.ylabel('Прогноз')
plt.title('Прогноз против настоящей стоимости')
plt.tight_layout()

## Мой вывод после запуска

Заполнить своими числами: какая модель победила, насколько она лучше baseline, где сильнее ошибается. Главные ограничения: данные 1990 года, обрезанная верхняя граница цены, пространственная зависимость. Для отдельного воспроизводимого запуска есть `python -m housing.train`.